In [1]:
import sys
import numpy as np
import tensorflow as tf
import tensorflow.keras as K
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

2026-01-28 16:59:26.774957: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-28 16:59:26.850058: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-28 16:59:29.025678: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-28 16:59:40.660631: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [2]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [3]:
layers = [4096, 2048, 1]
epochs = 50
act_func = tf.nn.relu
dropout = 0.5
input_dropout = 0.2
eta = 1e-5
norm = 'tanh'

In [4]:
X_tr, X_val, _, _, y_tr, y_val, _, _ = load(norm=norm)
print("Training data shape:", X_tr.shape)
print("Validation data shape:", X_val.shape)
print("NaN in X_tr:", np.isnan(X_tr).any())
print("NaN in y_tr:", np.isnan(y_tr).any())
print("Inf in X_tr:", np.isinf(X_tr).any())
print("Inf in y_tr:", np.isinf(y_tr).any())

Training data shape: (13884, 7063)
Validation data shape: (4614, 7063)
NaN in X_tr: False
NaN in y_tr: False
Inf in X_tr: False
Inf in y_tr: False


In [ ]:
model = Sequential()
for i in range(len(layers)):
    if i == 0:
        model.add(Dense(
            layers[i],
            input_shape=(X_tr.shape[1],),
            activation=act_func,
            kernel_initializer='he_normal'
        ))
        model.add(Dropout(float(input_dropout)))
    elif i == len(layers) - 1:
        model.add(Dense(
            layers[i],
            activation='linear',
            kernel_initializer="he_normal"
        ))
    else:
        model.add(Dense(
            layers[i],
            activation=act_func,
            kernel_initializer="he_normal"
        ))
        model.add(Dropout(float(dropout)))

In [ ]:
model.compile(
    loss='mean_squared_error',
    optimizer=K.optimizers.SGD(
        learning_rate=float(eta),
        momentum=0.5
    )
)
model.summary()

In [ ]:
hist = model.fit(
    X_tr, y_tr,
    epochs=epochs,
    batch_size=64,
    shuffle=True,
    validation_data=(X_val, y_val),
    verbose=1   
)

In [ ]:
val_loss = hist.history['val_loss']
train_loss = hist.history['loss']
print("Final training loss:", train_loss[-1])
print("Final validation loss:", val_loss[-1])
model.reset_states()